<a href="https://colab.research.google.com/github/cyberviser/Hancock/blob/main/Hancock_Universal_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyberviser/Hancock/blob/main/Hancock_Universal_Finetune.ipynb)

# 🔐 Hancock Universal Fine-Tuning — CyberViser
**Mistral 7B → Cybersecurity specialist via LoRA**

Works on: **Google Colab** (free T4) · **Kaggle** (free T4, 30h/week) · **RunPod/Vast** · **Oracle Cloud**

| Step | Time | Notes |
|------|------|-------|
| Install deps | ~3 min | Unsloth + TRL |
| Load 4-bit model | ~2 min | 7B params, 4GB VRAM |
| Train (300 steps) | ~25 min | v3 dataset (5,670 samples) |
| Export GGUF Q4 | ~5 min | Ready for Ollama |

**Setup:**
- **Colab:** Runtime → Change runtime type → T4 GPU
- **Kaggle:** Settings → Accelerator → GPU T4 x2 · Internet → On
- **Optional:** Add `HF_TOKEN` secret to push model to HuggingFace Hub

In [90]:
import os

# Force correct environment detection and WORK_DIR assignment
if os.path.exists('/kaggle'):
    ENV = 'Kaggle'
    WORK_DIR = '/kaggle/working'
    print(f'✅ Detected Kaggle environment. Setting WORK_DIR to {WORK_DIR}')
elif os.path.exists('/content'):
    ENV = 'Colab'
    WORK_DIR = '/content'
    print(f'✅ Detected Colab environment. Setting WORK_DIR to {WORK_DIR}')
else:
    ENV = 'Local'
    WORK_DIR = os.getcwd()
    print(f'✅ Detected Local environment. Setting WORK_DIR to {WORK_DIR}')

# Ensure directory exists and switch to it
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'📂 Current working directory: {os.getcwd()}')

# List contents to verify training artifacts
print('\n📄 Files in WORK_DIR:')
!ls -F

✅ Detected Kaggle environment. Setting WORK_DIR to /kaggle/working
📂 Current working directory: /kaggle/working

📄 Files in WORK_DIR:
Hancock/  huggingface_tokenizers_cache/  unsloth_compiled_cache/


In [108]:
import os

print(f"Checking WORK_DIR fix...")
print(f"Expected: /kaggle/working (if on Kaggle)")
print(f"Actual WORK_DIR: {WORK_DIR}")
print(f"Actual CWD: {os.getcwd()}")

print(f'\nDirectory contents:')
!ls -F

Checking WORK_DIR fix...
Expected: /kaggle/working (if on Kaggle)
Actual WORK_DIR: /kaggle/working
Actual CWD: /kaggle/working

Directory contents:
Hancock/  huggingface_tokenizers_cache/  unsloth_compiled_cache/


In [ ]:
import time
import os
from IPython.display import Audio, display

# This cell will run automatically AFTER the export cell finishes
print("\n" + "="*40)
print("✅  EXPORT CELL FINISHED!")
print("="*40 + "\n")

# Check for the GGUF file
gguf_file = "hancock-v3-unsloth.Q4_K_M.gguf"
if os.path.exists(gguf_file):
    size_mb = os.path.getsize(gguf_file) / (1024 * 1024)
    print(f"🎉 SUCCESS! Found GGUF file: {gguf_file} ({size_mb:.2f} MB)")
    print("You can now download it using the next cell.")
else:
    print("❌ GGUF file not found. The export might have failed.")

# Optional: Play a sound
try:
    display(Audio(url='https://www.soundjay.com/buttons/beep-01a.mp3', autoplay=True))
except:
    pass

In [ ]:
# @title 🛠️ Manual GGUF Export
from unsloth import FastLanguageModel
import os

# Use the path confirmed by the previous scan
# This ensures we don't get 'file not found' errors
adapter_path = os.path.join(WORK_DIR, 'Hancock', 'hancock-adapter-v3')

print(f"🎯 Targeted Adapter Path: {adapter_path}")

if not os.path.exists(adapter_path):
    print("❌ Adapter path does not exist. Please check the 'Scan' output above.")
else:
    print(f"🔄 Loading model from: {adapter_path}...")
    try:
        # 1. Load the model (4-bit to save memory)
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = adapter_path,
            max_seq_length = 4096,
            dtype = None,
            load_in_4bit = True,
        )

        # 2. Export to GGUF
        print("💾 Starting GGUF export (q4_k_m)... This may take 2-5 minutes.")
        model.save_pretrained_gguf("hancock-v3", tokenizer, quantization_method = "q4_k_m")
        print("✅ Export Complete! File saved as 'hancock-v3-unsloth.Q4_K_M.gguf'")

    except Exception as e:
        print(f"\n❌ Export Failed: {e}")
        print("Tip: If this is an OOM (Out of Memory) error, try restarting the runtime (Runtime > Restart Session) and running ONLY this cell.")

In [ ]:
import time
from IPython.display import Audio, display
import os

# This cell will only run AFTER the training cell finishes
print("\n" + "="*40)
print("✅  FULL TRAINING COMPLETED!")
print("="*40 + "\n")

# Check if the output files exist now
if 'WORK_DIR' in locals():
    adapters = [d for d in os.listdir(WORK_DIR) if 'adapter' in d]
    ggufs = [f for f in os.listdir(WORK_DIR) if f.endswith('.gguf')]
    print(f"Found {len(adapters)} adapter(s) and {len(ggufs)} GGUF file(s).")
else:
    print("WORK_DIR not defined, check manually.")

# Optional: Play a sound notification
try:
    display(Audio(url='https://www.soundjay.com/buttons/beep-01a.mp3', autoplay=True))
except:
    pass

In [ ]:
import os

# Define the path found in the previous scan
target_dir = os.path.join(WORK_DIR, 'Hancock', 'hancock-adapter-v3')

print(f"Inspecting contents of: {target_dir}")

if os.path.exists(target_dir):
    files = os.listdir(target_dir)
    if files:
        print(f"Files found: {files}")
        if 'adapter_config.json' in files:
            print("✅ SUCCESS: 'adapter_config.json' found! The debug training saved correctly.")
        else:
            print("❌ 'adapter_config.json' is still missing. Training failed to save.")
    else:
        print("❌ The directory is empty.")
else:
    print(f"❌ Directory does not exist: {target_dir}")

In [285]:
import time
from IPython.display import Audio, display

# This cell will only run AFTER the training cell finishes
print("\n" + "="*40)
print("✅  STEP 5 FINISHED: Training Completed!")
print("="*40 + "\n")

# Check if the output files exist now
import os
if 'WORK_DIR' in locals():
    adapters = [d for d in os.listdir(WORK_DIR) if 'adapter' in d]
    ggufs = [f for f in os.listdir(WORK_DIR) if f.endswith('.gguf')]
    print(f"Found {len(adapters)} adapter(s) and {len(ggufs)} GGUF file(s).")
else:
    print("WORK_DIR not defined, check manually.")

# Optional: Play a sound notification (works in some browsers)
try:
    display(Audio(url='https://www.soundjay.com/buttons/beep-01a.mp3', autoplay=True))
except:
    pass


✅  STEP 5 FINISHED: Training Completed!

Found 0 adapter(s) and 0 GGUF file(s).


In [261]:
import os

print("Checking for GGUF files in WORK_DIR...")
gguf_files = [f for f in os.listdir(WORK_DIR) if f.endswith('.gguf')]

if gguf_files:
    print(f"✅ Found GGUF file(s): {gguf_files}")
    print("Export is complete! You can now download the model.")
else:
    print("⏳ No GGUF files found yet.")
    print("The training or export process might still be running.")

Checking for GGUF files in WORK_DIR...
⏳ No GGUF files found yet.
The training or export process might still be running.


In [ ]:
import os

print("Checking for training completion...")

# Define expected output paths based on our fix
adapter_path = os.path.join(WORK_DIR, 'hancock-adapter-v3')

if os.path.exists(adapter_path):
    print(f"\n✅ Training Complete! Adapter found at: {adapter_path}")

    # Check for GGUF
    gguf_files = [f for f in os.listdir(WORK_DIR) if f.endswith('.gguf')]
    if gguf_files:
        print(f"✅ GGUF Model found: {gguf_files[0]}")
    else:
        print("⚠️ Adapter found, but no GGUF file yet (export might still be running).")
else:
    print("❌ No adapter found yet. If the training cell has finished, check its output for errors.")

In [465]:
import os

print(f"📂 Scanning {WORK_DIR} recursively for artifacts...")
found_adapter = False
found_gguf = False

for root, dirs, files in os.walk(WORK_DIR):
    # Check for adapter dir
    for name in dirs:
        if 'hancock' in name.lower() and 'adapter' in name.lower():
            print(f"✅ Found Adapter: {os.path.join(root, name)}")
            found_adapter = True
    # Check for GGUF files
    for name in files:
        if name.endswith('.gguf'):
            print(f"✅ Found GGUF: {os.path.join(root, name)}")
            found_gguf = True

if not found_adapter and not found_gguf:
    print("❌ No model artifacts found. The training likely failed.")
    print("👉 Please check the output of Cell 5 (the training cell) for error messages.")
else:
    print("\nSearch complete.")

📂 Scanning /content recursively for artifacts...
✅ Found Adapter: /content/Hancock/hancock-adapter-v3
✅ Found Adapter: /content/Hancock/hancock-cpu-adapter

Search complete.


In [89]:
# @title 2️⃣  Install Dependencies (~3 min)
import subprocess, sys

# Install stable release from PyPI to avoid unstable git changes
if ENV == 'Kaggle':
    !pip install "unsloth[kaggle-new]" -q
else:
    !pip install "unsloth[colab-new]" -q

!pip install -q 'trl>=0.8.6' 'transformers>=4.40' 'datasets>=2.18' peft huggingface_hub accelerate bitsandbytes

import importlib, pkg_resources
for pkg in ['unsloth','trl','transformers','datasets','peft']:
    try:
        v = pkg_resources.get_distribution(pkg).version
    except pkg_resources.DistributionNotFound:
        v = 'Not Found'
    print(f'  {pkg}: {v}')
print('\n✅ Dependencies installed')

  unsloth: 2026.2.1
  trl: 0.24.0
  transformers: 4.57.6
  datasets: 4.3.0
  peft: 0.18.1

✅ Dependencies installed


In [78]:
# @title 3️⃣  Clone Hancock & Check GPU
import torch
import os

# Clone repo
hancock_dir = os.path.join(WORK_DIR, 'Hancock')
if not os.path.exists(hancock_dir):
    print(f'Cloning Hancock repo to {hancock_dir}...')
    !git clone https://github.com/cyberviser/Hancock.git {hancock_dir}
else:
    print(f'Hancock repo already exists at {hancock_dir}')

os.chdir(hancock_dir)

# GPU check
assert torch.cuda.is_available(), '❌ Enable GPU runtime first!'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
print(f'Repo: {os.getcwd()}')
print('✅ Ready')

Hancock repo already exists at /kaggle/working/Hancock
GPU: Tesla T4 | VRAM: 15.6 GB
Repo: /kaggle/working/Hancock
✅ Ready


In [82]:
# @title 4️⃣  Configure Training
# Adjust these settings before running:

TRAINING_STEPS = 300        # More steps = better quality, longer time (~5 min per 100 steps on T4)
EXPORT_GGUF = True          # Export GGUF for Ollama deployment
PUSH_TO_HUB = False         # Push to HuggingFace Hub (requires HF_TOKEN)
LORA_RANK = 0               # 0 = auto-detect based on VRAM (16=T4, 32=24GB, 64=40GB+)

print(f'Steps: {TRAINING_STEPS}')
print(f'GGUF export: {EXPORT_GGUF}')
print(f'Push to Hub: {PUSH_TO_HUB}')
print(f'LoRA rank: {"auto" if LORA_RANK == 0 else LORA_RANK}')

Steps: 300
GGUF export: True
Push to Hub: False
LoRA rank: auto


In [449]:
# @title 5️⃣  Run Fine-Tuning (~25 min on T4)
# Uses the universal hancock_finetune_v3.py script
import os

# Force disable WANDB to prevent logging errors
%env WANDB_DISABLED=true

# Ensure TRAINING_STEPS is defined (default to 300 if missing from previous cells)
if 'TRAINING_STEPS' not in locals():
    TRAINING_STEPS = 300

# Locate the script
script_name = 'hancock_finetune_v3.py'
script_path = os.path.join('Hancock', script_name) if os.path.exists(os.path.join('Hancock', script_name)) else script_name

# RUN FULL TRAINING
print(f"🚀 STARTING FULL TRAINING: {TRAINING_STEPS} steps")

cmd = f'python {script_path} --steps {TRAINING_STEPS}'
if 'EXPORT_GGUF' in locals() and EXPORT_GGUF:
    cmd += ' --export-gguf'
if 'PUSH_TO_HUB' in locals() and PUSH_TO_HUB:
    cmd += ' --push-to-hub'
if 'LORA_RANK' in locals() and LORA_RANK > 0:
    cmd += f' --lora-r {LORA_RANK}'

print(f'Running: {cmd}\n')
!{cmd}

env: WANDB_DISABLED=true
🚀 STARTING FULL TRAINING: 300 steps
Running: python hancock_finetune_v3.py --steps 300 --export-gguf


[v3] Hancock Fine-Tune v3 — CyberViser
     Platform : Linux (Colab)
     GPU      : Tesla T4
     VRAM     : 15.6 GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-02 22:46:17.940449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772491577.961393  222617 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772491577.968274  222617 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772491577.985670  222617 computation_placer.cc:177] computation placer already registered. Please check l

In [322]:
# @title 6️⃣  Test the Fine-Tuned Model
import torch
from unsloth import FastLanguageModel
import os

# Robustly find the adapter path
adapter_path = None
possible_roots = [os.getcwd(), WORK_DIR, os.path.join(WORK_DIR, 'Hancock')]

# First check specific locations
for root in possible_roots:
    potential = os.path.join(root, 'hancock-adapter-v3')
    if os.path.exists(potential) and 'adapter_config.json' in os.listdir(potential):
        adapter_path = potential
        break

# Fallback: Recursive search if not found yet
if not adapter_path:
    for root, dirs, files in os.walk(WORK_DIR):
        if 'hancock-adapter-v3' in dirs:
             p = os.path.join(root, 'hancock-adapter-v3')
             if 'adapter_config.json' in os.listdir(p):
                 adapter_path = p
                 break

if adapter_path:
    print(f"✅ Found adapter at: {adapter_path}")
    print(f"📂 Files in adapter folder: {os.listdir(adapter_path)}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_path,
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    messages = [
        {'role': 'system', 'content': 'You are Hancock, an elite cybersecurity AI by CyberViser.'},
        {'role': 'user', 'content': 'Explain CVE-2021-44228 Log4Shell and provide a Splunk SPL query to detect exploitation attempts.'},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=512,
            use_cache=True, temperature=0.7, do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print('\n🛡️ Hancock says:')
    print('=' * 60)
    print(response)
else:
    print("❌ Critical Error: Could not find 'hancock-adapter-v3' containing 'adapter_config.json'.")
    print("Please check if the training (Step 5) completed successfully.")

❌ Critical Error: Could not find 'hancock-adapter-v3' containing 'adapter_config.json'.
Please check if the training (Step 5) completed successfully.


In [104]:
# @title 7️⃣  Download Model Files
import os
from pathlib import Path

print('\n📦 Available model files:\n')

# LoRA adapter
adapter_dir = Path('hancock-adapter-v3')
if adapter_dir.exists():
    size = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
    print(f'  📁 hancock-adapter-v3/ ({size:.0f} MB) — LoRA adapter')

# GGUF
for gguf in Path('.').glob('*.gguf'):
    size = gguf.stat().st_size / 1e6
    print(f'  📁 {gguf.name} ({size:.0f} MB) — GGUF for Ollama')

print('\n⬇️  Download options:')
if ENV == 'Colab':
    print('  Option 1: Run the cell below to download via browser')
    print('  Option 2: Mount Google Drive and copy there')
elif ENV == 'Kaggle':
    print('  Option 1: Output tab → download files from /kaggle/working/Hancock/')
    print('  Option 2: Push to HuggingFace Hub (re-run with PUSH_TO_HUB=True)')
else:
    print('  Files are saved in the current directory')


📦 Available model files:


⬇️  Download options:
  Option 1: Run the cell below to download via browser
  Option 2: Mount Google Drive and copy there


In [ ]:
# @title 8️⃣  Download GGUF (Colab only)
# Skip this cell on Kaggle — use Output tab instead
import os

try:
    from google.colab import files
    for f in os.listdir('.'):
        if f.endswith('.gguf'):
            print(f'Downloading {f}...')
            files.download(f)
            break
    else:
        print('No GGUF file found. Run with EXPORT_GGUF=True')
except ImportError:
    print('Not running on Colab — download manually from the Output tab (Kaggle) or local filesystem')

---
## 🚀 Deploy with Ollama (after download)

```bash
# On your local machine:
mkdir -p ~/cyberviser/models && mv hancock-v3-Q4_K_M.gguf ~/cyberviser/models/
cd ~/cyberviser/Hancock

# Update Modelfile to point to GGUF:
# FROM ./models/hancock-v3-Q4_K_M.gguf

ollama create hancock-finetuned -f Modelfile.hancock-finetuned
ollama run hancock-finetuned
```

---
© 2026 CyberViser · [cyberviser.netlify.app](https://cyberviser.netlify.app)